# 第 14 讲：敏感性、鲁棒性与模型检验

> 对模型关键参数做扰动，检查结论是否稳定。

## 课件导学

**任务情境**：模型结论必须经得起参数扰动，本讲训练敏感性分析和鲁棒性表达。

**关键概念**

- 参数扰动
- 敏感性曲线
- 解结构稳定性
- 鲁棒性说明

**本讲产物**

- lp_sensitivity.csv
- lp_sensitivity.png
- robustness_statement.csv

## 资料链接与阅读抓手

下面这些链接优先选官方文档或稳定资料。课前先看标题和示例，课堂中遇到参数或概念再回查。

- [SciPy linprog 文档](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.linprog.html)：复用线性规划模型作为敏感性对象。
- [scikit-learn 模型评估](https://scikit-learn.org/stable/modules/model_evaluation.html)：类比理解评价指标与模型检验。
- [Matplotlib pyplot 教程](https://matplotlib.org/stable/tutorials/pyplot.html)：将参数变化画成可解释曲线。

## 使用说明

- 先从上到下运行全部单元。
- 每讲都会生成一个 `outputs/lesson-xx/` 目录，用于保存图表、表格或报告片段。
- 示例数据均为教学用合成数据，真实比赛中应替换为题目附件数据。

In [ ]:
LESSON_ID = "lesson-14"

from pathlib import Path
import os
import math
import itertools
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

OUTPUT_ROOT = Path(os.environ.get("COURSE_OUTPUT_DIR", REPO_ROOT / "outputs")) / LESSON_ID
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(42)
print(f"Output directory: {OUTPUT_ROOT}")

## 可运行示例与讲解路线

In [ ]:
from scipy.optimize import linprog

profits = np.arange(3.0, 6.1, 0.25)
rows = []
for p2 in profits:
    res = linprog([-3, -p2], A_ub=[[2, 1], [1, 3], [0, 1]], b_ub=[100, 150, 40], bounds=[(0, None), (0, None)], method="highs")
    rows.append({"profit_B": p2, "x1": res.x[0], "x2": res.x[1], "objective": -res.fun})
sensitivity = pd.DataFrame(rows)
sensitivity.to_csv(OUTPUT_ROOT / "lp_sensitivity.csv", index=False)
sensitivity.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(sensitivity["profit_B"], sensitivity["x1"], label="x1")
axes[0].plot(sensitivity["profit_B"], sensitivity["x2"], label="x2")
axes[0].set_title("Decision variables under profit_B changes")
axes[0].legend()
axes[1].plot(sensitivity["profit_B"], sensitivity["objective"], color="darkgreen")
axes[1].set_title("Objective sensitivity")
for ax in axes:
    ax.set_xlabel("profit_B")
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "lp_sensitivity.png", dpi=160)
plt.show()

In [ ]:
stable_choice = sensitivity[["x1", "x2"]].round(3).drop_duplicates().shape[0] <= 3
pd.DataFrame({"结论": ["模型在利润参数变化下的解结构是否稳定"], "是否稳定": [stable_choice]}).to_csv(
    OUTPUT_ROOT / "robustness_statement.csv", index=False, encoding="utf-8-sig"
)
stable_choice

## 实战环节

**课堂任务**

- 选择一个关键参数做 10 个扰动点。
- 判断最优决策是否发生跳变。
- 写一段鲁棒性结论。

**课后挑战**：同时扰动两个参数，输出热力图或二维结果表。

**验收清单**

- 扰动范围是否合理
- 是否说明结论稳定区间
- 是否把检验结果写进论文语气

In [ ]:
lesson_resources = [{'title': 'SciPy linprog 文档', 'url': 'https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.linprog.html', 'reading_note': '复用线性规划模型作为敏感性对象。'}, {'title': 'scikit-learn 模型评估', 'url': 'https://scikit-learn.org/stable/modules/model_evaluation.html', 'reading_note': '类比理解评价指标与模型检验。'}, {'title': 'Matplotlib pyplot 教程', 'url': 'https://matplotlib.org/stable/tutorials/pyplot.html', 'reading_note': '将参数变化画成可解释曲线。'}]
lesson_deliverables = ['lp_sensitivity.csv', 'lp_sensitivity.png', 'robustness_statement.csv']
lesson_checklist = ['扰动范围是否合理', '是否说明结论稳定区间', '是否把检验结果写进论文语气']

pd.DataFrame(lesson_resources).to_csv(OUTPUT_ROOT / "lesson_resources.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"deliverable": lesson_deliverables}).to_csv(OUTPUT_ROOT / "lesson_deliverables.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"check_item": lesson_checklist}).to_csv(OUTPUT_ROOT / "lesson_checklist.csv", index=False, encoding="utf-8-sig")
print("Saved courseware resource map, deliverables, and checklist.")